# Poem Sentiment: Ministral-3-8B few-Shot + QLoRA Fine-Tuning

**Comparison plan:**
- Naive Bayes baseline: **66.35%** (predicts only `no_impact`)
- Ministral-3-8B zero-shot (NF4)
- Ministral-3-8B fine-tuned with QLoRA

**Label map:** `0=negative`, `1=positive`, `2=no_impact`, `3=mixed`

In [ ]:
# Install dependencies (run once)
# !pip install "transformers>=5.0.0rc0" "mistral-common>=1.8.6" accelerate bitsandbytes peft trl datasets -q


In [ ]:
import json
import re
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    Mistral3ForConditionalGeneration,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
from trl import SFTTrainer, SFTConfig
from sklearn import metrics

MODEL_ID = "unsloth/Ministral-3-8B-Base-2512-unsloth-bnb-4bit"
LABEL_MAP = {0: "negative", 1: "positive", 2: "no_impact", 3: "mixed"}
LABEL_MAP_INV = {v: k for k, v in LABEL_MAP.items()}

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

## 1. Load Dataset

In [ ]:
ds = load_dataset("google-research-datasets/poem_sentiment")

for split, subset in ds.items():
    from collections import Counter
    label_counts = Counter(subset['label'])
    readable = {LABEL_MAP[k]: v for k, v in sorted(label_counts.items())}
    print(f"{split:12s} n={len(subset):4d}  {readable}")

## 2. Load Model (4 bits Quantization)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Pre-quantized weights already in 4-bit — no quantization_config needed
model = Mistral3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
)
model.config.use_cache = False

print(f"Model loaded. Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## 3. Few-Shot Inference

Base model = text completion, not instruction following.  
We use **few-shot prompting** so the natural completion is a JSON label.

In [ ]:
FEW_SHOT_EXAMPLES = [
    ("and the dead leaves lie huddled and still", "negative"),
    ("what light through yonder window breaks", "positive"),
    ("the sun sets slowly in the west", "no_impact"),
    ("a tender joy yet touched with pain", "mixed"),
]

def build_prompt(verse: str, include_answer: bool = False) -> str:
    """Few-shot prompt for a base model. If include_answer=True, appends the
    closing JSON — used for building fine-tuning examples."""
    lines = []
    lines.append("Classify the sentiment of each poem verse. "
                 "Output only a JSON object with key 'label'. "
                 "Valid values: negative, positive, no_impact, mixed.\n")
    for text, label in FEW_SHOT_EXAMPLES:
        lines.append(f'Verse: "{text}"')
        lines.append(f'Output: {{"label": "{label}"}}\n')
    lines.append(f'Verse: "{verse}"')
    lines.append('Output: {')
    if include_answer:
        # For fine-tuning SFT format we append the completion too
        pass  # handled separately below
    return "\n".join(lines)

# Sanity check
print(build_prompt("the morning dew on silent grass"))

In [ ]:
def parse_label(raw_output: str) -> tuple[int | None, str]:
    """
    Extract label from model output.
    Returns (label_int_or_None, status) where status is 'ok' or 'parse_error'.
    The prompt ends with '{' so we prepend it before parsing.
    """
    # Grab only the first line of the completion (stop at newline)
    first_line = raw_output.strip().split("\n")[0]
    candidate = "{" + first_line  # re-attach the opening brace

    # Try strict JSON first
    try:
        obj = json.loads(candidate)
        label_str = obj.get("label", "").strip().lower()
        if label_str in LABEL_MAP_INV:
            return LABEL_MAP_INV[label_str], "ok"
    except json.JSONDecodeError:
        pass

    # Fallback: regex scan for any valid label word
    for label_str, label_int in LABEL_MAP_INV.items():
        if re.search(label_str, candidate, re.IGNORECASE):
            return label_int, "regex_fallback"

    return None, "parse_error"

In [ ]:
@torch.inference_mode()
def predict_few_shot(verse: str) -> tuple[int | None, str, str]:
    """Returns (predicted_label_int, raw_completion, parse_status)"""
    prompt = build_prompt(verse)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[1]

    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        max_length=None,       # suppress conflict with model's default max_length
        do_sample=False,       # greedy — deterministic
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    completion = tokenizer.decode(
        output_ids[0][prompt_len:], skip_special_tokens=True
    )
    label, status = parse_label(completion)
    return label, completion, status

# Quick smoke test
lbl, raw, status = predict_few_shot("with pale blue berries. in these peaceful shades--")
print(f"Predicted: {LABEL_MAP.get(lbl, 'NONE')} | status={status} | raw='{raw}'")

## 4. Few-Shot Evaluation on Test Set

In [ ]:
from tqdm.notebook import tqdm

test_texts  = [x['verse_text'] for x in ds['test']]
test_labels = [x['label']      for x in ds['test']]

zs_predictions = []
zs_statuses    = []

for verse in tqdm(test_texts, desc="Few-shot inference"):
    lbl, _, status = predict_few_shot(verse)
    zs_predictions.append(lbl)
    zs_statuses.append(status)

print("Parse status distribution:", Counter(zs_statuses))
zs_pred_clean = [p if p is not None else 2 for p in zs_predictions]

zs_acc = metrics.accuracy_score(test_labels, zs_pred_clean)
print(f"\nFew-shot accuracy: {zs_acc:.4f}")

all_labels = sorted(LABEL_MAP.keys())
print(metrics.classification_report(
    test_labels, zs_pred_clean,
    labels=all_labels,
    target_names=[LABEL_MAP[i] for i in all_labels],
    zero_division=0
))

## 5. QLoRA Fine-Tuning Setup

In [ ]:
def make_sft_example(verse: str, label_int: int) -> str:
    """Format one training example as prompt + completion for SFT."""
    prompt = build_prompt(verse)
    label_str = LABEL_MAP[label_int]
    # prompt already ends with 'Output: {', complete the JSON
    return prompt + f'"label": "{label_str}"}}'

# Preview one example
ex = ds['train'][0]
print(make_sft_example(ex['verse_text'], ex['label']))

In [ ]:
from datasets import Dataset

def build_sft_dataset(split) -> Dataset:
    texts = [make_sft_example(x['verse_text'], x['label']) for x in split]
    return Dataset.from_dict({"text": texts})

train_sft = build_sft_dataset(ds['train'])
val_sft   = build_sft_dataset(ds['validation'])

print(f"SFT train: {len(train_sft)}  val: {len(val_sft)}")
print(train_sft[0]['text'][:200], "...")

In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                          # rank
    lora_alpha=32,                 # scaling = alpha/r = 2
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Train

In [ ]:
from trl import SFTConfig

sft_config = SFTConfig(
    output_dir="./ministral-poem-lora",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=True,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="best",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_8bit",
    max_length=512,
    packing=False,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_sft,
    eval_dataset=val_sft,
    processing_class=tokenizer,
)

trainer.train()

## 7. Post-Fine-Tuning Evaluation

In [ ]:
model.eval()

ft_predictions = []
ft_statuses    = []

for verse in tqdm(test_texts, desc="Fine-tuned inference"):
    lbl, _, status = predict_few_shot(verse)
    ft_predictions.append(lbl)
    ft_statuses.append(status)

print("Parse status distribution:", Counter(ft_statuses))
ft_pred_clean = [p if p is not None else 2 for p in ft_predictions]

ft_acc = metrics.accuracy_score(test_labels, ft_pred_clean)
print(f"\nFine-tuned accuracy: {ft_acc:.4f}")

all_labels = sorted(LABEL_MAP.keys())
print(metrics.classification_report(
    test_labels, ft_pred_clean,
    labels=all_labels,
    target_names=[LABEL_MAP[i] for i in all_labels],
    zero_division=0
))

## 8. Final Comparison

In [ ]:
NAIVE_BAYES_ACC = 0.6635

print("=" * 45)
print(f"{'Model':<30} {'Accuracy':>10}")
print("=" * 45)
print(f"{'Naive Bayes (TF-IDF)':<30} {NAIVE_BAYES_ACC:>10.4f}")
print(f"{'Ministral-3-8B few-shot':<30} {zs_acc:>10.4f}")
print(f"{'Ministral-3-8B QLoRA':<30} {ft_acc:>10.4f}")
print("=" * 45)
print()
print("NOTE: parse errors in few-shot vs fine-tuned:")
print(f"  few-shot parse errors: {zs_statuses.count('parse_error')}")
print(f"  fine-tuned parse errors: {ft_statuses.count('parse_error')}")